# Sprint 4K Image-Level Balanced Sampler

Thin Colab runner for the Sprint 4K diagnostic. Core logic stays in `src/dl_midterm/` and `scripts/`. This run tests image-level class-balanced sampling during CNN fine-tuning, starting with ResNet50 `layer3 + layer4` under a conservative learning-rate policy. Full operational artifacts are mirrored to Drive without excluding checkpoints, feature caches, or MLP `model.pt` files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -d dl-assignment/.git ]; then
  git clone https://github.com/KasimDeliaci/dl-midterm-project.git dl-assignment
fi
cd /content/dl-assignment
git pull --ff-only
python -m pip install -q uv
uv sync
mkdir -p /content/drive/MyDrive/dl-midterm-artifacts/sprint4k

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
BUNDLE=""
for candidate in \
  /content/drive/MyDrive/ham10000_colab_bundle.tar \
  "/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar" \
  /content/drive/MyDrive/dl-assignment/ham10000_colab_bundle.tar; do
  if [ -f "$candidate" ]; then BUNDLE="$candidate"; break; fi
done
if [ -z "$BUNDLE" ]; then
  echo "HAM10000 bundle not found on Drive" >&2
  exit 1
fi
tar -xf "$BUNDLE" -C /content/dl-assignment
echo "Restored HAM10000 bundle from: $BUNDLE"
echo "Split rows:"
wc -l data/splits/train.csv data/splits/val.csv data/splits/test.csv

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python - <<'PY'
from pathlib import Path
import yaml

config = yaml.safe_load(Path('configs/experiments/sprint4k_image_balanced_sampler_backbones.yaml').read_text())
ft = config['finetuning']
print('Experiment:', ft['experiment_name'])
print('Feature source:', ft['feature_source'])
print('Sampler:', ft.get('sampler'))
print('Class weighting:', ft.get('class_weighting'))
print('ResNet50 unfreeze policy:', ft['unfreeze_policies']['resnet50'])
print('Epochs:', ft['epochs'])
print('Backbone LR:', ft['backbone_learning_rate'])
print('Head LR:', ft['head_learning_rate'])
PY

## ResNet50 Diagnostic

Run this first. It fine-tunes only ResNet50 with `layer3 + layer4` unfrozen, class-balanced image sampling enabled, and class weights disabled.

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
PYTHONUNBUFFERED=1 uv run python scripts/finetune_backbone.py \
  --config configs/experiments/sprint4k_image_balanced_sampler_backbones.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbone resnet50 \
  --feature-source finetuned_balanced_sampler \
  --batch-size 32

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
DEST=/content/drive/MyDrive/dl-midterm-artifacts/sprint4k
mkdir -p \
  "$DEST/checkpoints/finetuned_balanced_sampler_backbones" \
  "$DEST/features/ham10000/finetuned_balanced_sampler" \
  "$DEST/runs" \
  "$DEST/report_assets"
rsync -a artifacts/checkpoints/finetuned_balanced_sampler_backbones/ \
  "$DEST/checkpoints/finetuned_balanced_sampler_backbones/"
rsync -a artifacts/features/ham10000/finetuned_balanced_sampler/ \
  "$DEST/features/ham10000/finetuned_balanced_sampler/"
rsync -a artifacts/runs/ \
  "$DEST/runs/"
rsync -a artifacts/report_assets/ \
  "$DEST/report_assets/"
echo "Backbone checkpoints on Drive:"
find "$DEST/checkpoints" -name '*.pt' | sort | wc -l
echo "Feature cache .pt files on Drive:"
find "$DEST/features" -name '*.pt' | sort | wc -l
echo "MLP model.pt files on Drive:"
find "$DEST/runs" -name 'model.pt' | sort | wc -l

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python - <<'PY'
import json
from pathlib import Path

summary_path = Path('artifacts/runs/sprint4_finetune_backbones_summary.json')
runs = []
if summary_path.exists():
    for row in json.loads(summary_path.read_text()):
        if row.get('backbone') == 'resnet50' and row.get('run_dir'):
            path = Path(row['run_dir']) / 'metrics.json'
            if path.exists():
                runs.append(path)
if not runs:
    runs = sorted(Path('artifacts/runs/finetune_backbones').glob('*finetuned_balanced_sampler*/metrics.json'))
if not runs:
    runs = sorted(Path('artifacts/runs/finetune_backbones').glob('*/metrics.json'))
if not runs:
    raise SystemExit('No fine-tuning metrics.json found yet.')
path = runs[-1]
metrics = json.loads(path.read_text())
print('Metrics file:', path)
print('best_val_epoch:', metrics.get('best_val_epoch'))
print('best_val_macro_f1:', metrics.get('best_val_macro_f1'))
print('test_accuracy:', metrics.get('accuracy'))
print('test_macro_f1:', metrics.get('macro_f1'))
print('test_weighted_f1:', metrics.get('weighted_f1'))
PY

## Optional Full Matrix

Only run this if the ResNet50 diagnostic is promising. Set `RUN_FULL_MATRIX=1` inside the next cell. It trains all configured backbones, extracts feature caches, then runs the cached-feature single-backbone and fusion MLP matrix.

In [ ]:
%%bash
set -euo pipefail
RUN_FULL_MATRIX=0
if [ "$RUN_FULL_MATRIX" != "1" ]; then
  echo "Skipping optional full Sprint 4K matrix. Set RUN_FULL_MATRIX=1 in this cell to run it."
  exit 0
fi
cd /content/dl-assignment
PYTHONUNBUFFERED=1 uv run python scripts/finetune_backbone.py \
  --config configs/experiments/sprint4k_image_balanced_sampler_backbones.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_balanced_sampler \
  --batch-size 32
PYTHONUNBUFFERED=1 uv run python scripts/train_mlp.py \
  --config configs/experiments/sprint4k_balanced_sampler_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_balanced_sampler \
  --experiment-name sprint4k_balanced_sampler_single_backbone
PYTHONUNBUFFERED=1 uv run python scripts/run_experiment_matrix.py \
  --config configs/experiments/sprint4k_balanced_sampler_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_balanced_sampler \
  --experiment-name sprint4k_balanced_sampler_feature_matrix

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
DEST=/content/drive/MyDrive/dl-midterm-artifacts/sprint4k
mkdir -p \
  "$DEST/checkpoints/finetuned_balanced_sampler_backbones" \
  "$DEST/features/ham10000/finetuned_balanced_sampler" \
  "$DEST/runs" \
  "$DEST/report_assets"
rsync -a artifacts/checkpoints/finetuned_balanced_sampler_backbones/ \
  "$DEST/checkpoints/finetuned_balanced_sampler_backbones/"
rsync -a artifacts/features/ham10000/finetuned_balanced_sampler/ \
  "$DEST/features/ham10000/finetuned_balanced_sampler/"
rsync -a artifacts/runs/ \
  "$DEST/runs/"
rsync -a artifacts/report_assets/ \
  "$DEST/report_assets/"
echo "Backbone checkpoints on Drive:"
find "$DEST/checkpoints" -name '*.pt' | sort | wc -l
echo "Feature cache .pt files on Drive:"
find "$DEST/features" -name '*.pt' | sort | wc -l
echo "MLP model.pt files on Drive:"
find "$DEST/runs" -name 'model.pt' | sort | wc -l
echo "Sprint 4K Drive artifact root: $DEST"